# Serve embedding trên Kaggle: vLLM + ngrok

**Chuẩn bị:** phiên Kaggle mới, GPU T4, Internet bật. Thêm hai Secrets và cấp quyền: `VLLM_API_KEY`, `NGROK_AUTHTOKEN`.

Chạy lần lượt các cell. Chỉ phục vụ model, không tạo Chroma.
Model, tokenizer và revision **97b0c614be4d77ee51c0cef4e5f07c00f9eb65b3** đã khóa trong `scripts/serve_embedding.sh` của repo.

Cell cuối kiểm tra URL ngrok rồi chờ trong lúc bạn dùng endpoint từ local. Nó **không bảo đảm giữ session**, không giả lập hoạt động hay gửi request định kỳ. Nếu Kaggle dừng phiên thì vLLM/ngrok cũng dừng. Dùng phiên tương tác cho demo; Save & Run All không tạo server vĩnh viễn.


## 1. Kiểm tra GPU và Python


In [ ]:
!nvidia-smi
!python --version


## 2. Cài môi trường riêng

Giữ PyTorch của notebook nguyên trạng. `virtualenv` và `wrapt` xử lý các lỗi môi trường đã gặp. Bước này tải thư viện, chưa tải model. Không chạy lại cell cài đặt khi server đang chạy.


In [ ]:
import subprocess
import sys
from pathlib import Path

ENV = Path("/kaggle/working/vllm-serve-env")
PYTHON = str(ENV / "bin/python")

subprocess.run([sys.executable, "-m", "pip", "install", "virtualenv", "pyngrok==8.1.2", "requests"], check=True)
if not Path(PYTHON).exists():
    subprocess.run([sys.executable, "-m", "virtualenv", str(ENV)], check=True)
subprocess.run([PYTHON, "-m", "pip", "install", "wrapt"], check=True)
subprocess.run([
    PYTHON, "-m", "pip", "install", "vllm==0.18.0", "torch==2.10.0+cu129",
    "--extra-index-url", "https://download.pytorch.org/whl/cu129",
], check=True)


## 3. Clone repo

Nếu repo đã có thì chỉ cập nhật. Không cài package dự án vì bước serve chỉ cần chạy file `.sh`.


In [ ]:
REPO = Path("/kaggle/working/Vietnamese_Legal_Rag")
REPO_URL = "https://github.com/nhannhan2201/Vietnamese_Legal_Rag.git"

if (REPO / ".git").is_dir():
    subprocess.run(["git", "-C", str(REPO), "pull", "--ff-only"], check=True)
else:
    subprocess.run(["git", "clone", REPO_URL, str(REPO)], check=True)


## 4. Đọc Secret và đặt đường dẫn CUDA

Không in khóa. `LD_LIBRARY_PATH` giúp chạy đúng CUDA, `LIBRARY_PATH` giúp FlashInfer biên dịch với driver NVIDIA. Tự tìm site-packages, không ghi cứng phiên bản Python.


In [ ]:
import os
from kaggle_secrets import UserSecretsClient

secrets = UserSecretsClient()
api_key = secrets.get_secret("VLLM_API_KEY")
ngrok_token = secrets.get_secret("NGROK_AUTHTOKEN")
if not api_key.strip() or not ngrok_token.strip():
    raise RuntimeError("Hai Secret phải có giá trị và được bật quyền truy cập.")

site_packages = subprocess.check_output(
    [PYTHON, "-c", "import sysconfig; print(sysconfig.get_paths()['purelib'])"], text=True
).strip().splitlines()[-1]

env = os.environ.copy()
# Dùng cấu hình model/revision/cổng mặc định đã khóa trong file .sh.
for name in ("MODEL", "MODEL_REVISION", "TOKENIZER_REVISION", "SERVED_MODEL_NAME",
             "HOST", "PORT", "MAX_MODEL_LEN", "GPU_MEMORY_UTILIZATION"):
    env.pop(name, None)
env["PYTHON_BIN"] = PYTHON
env["VLLM_API_KEY"] = api_key
env["CUDA_VISIBLE_DEVICES"] = "0"
env["VLLM_LIBRARY_PATH"] = site_packages + "/nvidia/nvjitlink/lib:/usr/local/nvidia/lib64"
env["LIBRARY_PATH"] = "/usr/local/nvidia/lib64"
print("Đã chuẩn bị môi trường và hai Secret.")


## 5. Chạy file .sh và chờ server

Chạy một lần. Script tải model cố định vào cache, nạp GPU rồi mở API. Không khởi động server thứ hai nếu cổng 8000 đang được dùng. Log ở `/kaggle/working/embedding-server.log`.


In [ ]:
import socket
import time
import requests

# Chặn việc vô tình khởi động thêm server trên cổng đang dùng.
with socket.socket() as port_check:
    try:
        port_check.bind(("127.0.0.1", 8000))
    except OSError:
        raise RuntimeError("Cổng 8000 đang được dùng. Không chạy lại cell khởi động server.") from None

LOG = Path("/kaggle/working/embedding-server.log")
with LOG.open("w") as log:
    server = subprocess.Popen(
        ["bash", "scripts/serve_embedding.sh"], cwd=str(REPO), env=env,
        stdout=log, stderr=subprocess.STDOUT, start_new_session=True,
    )
print("Server PID:", server.pid)

http = requests.Session()
http.trust_env = False
BASE_URL = "http://127.0.0.1:8000/v1"
headers = {"Authorization": f"Bearer {api_key}"}
for attempt in range(180):  # Chờ tối đa khoảng 15 phút khi tải/khởi động lần đầu.
    if server.poll() is not None:
        raise RuntimeError("Server đã dừng. Xem log:\n" + LOG.read_text(errors="replace")[-8000:])
    try:
        response = http.get(BASE_URL + "/models", headers=headers, timeout=(2, 3))
        if response.status_code == 401:
            raise RuntimeError("Server từ chối API key. Kiểm tra Secret và cấu hình khởi động.")
        if response.status_code == 200:
            break
    except requests.RequestException:
        pass
    if attempt % 12 == 0:
        print("Đang chờ server...", flush=True)
    time.sleep(5)
else:
    raise TimeoutError("Chưa sẵn sàng. Xem log; không chạy thêm server.")
print("Server sẵn sàng:", BASE_URL)


## 6. Thử embedding và API key

Thiếu key phải nhận 401; key đúng phải nhận vector 1024 chiều hợp lệ. Chỉ mở ngrok nếu cell này thành công.


In [ ]:
import math

payload = {
    "model": "Qwen/Qwen3-Embedding-0.6B",
    "input": ["Người lao động có quyền nghỉ hằng năm."],
    "encoding_format": "float",
}
without_key = http.post(BASE_URL + "/embeddings", json=payload, timeout=(5, 60))
if without_key.status_code != 401:
    raise RuntimeError("API chưa chặn request thiếu key; chưa mở ngrok.")

response = http.post(BASE_URL + "/embeddings", json=payload, headers=headers, timeout=(5, 180))
response.raise_for_status()
vector = response.json()["data"][0]["embedding"]
if len(vector) != 1024 or not all(math.isfinite(x) for x in vector):
    raise RuntimeError("Vector trả về không hợp lệ.")
norm = math.sqrt(sum(x*x for x in vector))
if abs(norm - 1) > 0.01:
    raise RuntimeError("Độ dài vector khác kỳ vọng.")
embedding_ok = True
print("Không key:", without_key.status_code, "| Có key:", response.status_code)
print("Số chiều:", len(vector), "| Độ dài vector:", norm)


## 7. Mở ngrok

Chạy một lần để tạo URL công khai. Không truyền ngrok authtoken cho client local; client chỉ dùng `VLLM_API_KEY`. Nếu chạy lại cell trong cùng kernel thì dùng lại tunnel đã mở.


In [ ]:
from pyngrok import conf, ngrok

if not embedding_ok or server.poll() is not None:
    raise RuntimeError("Server chưa sẵn sàng.")

ngrok_config = conf.PyngrokConfig(auth_token=ngrok_token)
active = ngrok.get_tunnels(pyngrok_config=ngrok_config)

old_tunnel = globals().get("tunnel")
active_urls = {item.public_url for item in active}

if old_tunnel is None or old_tunnel.public_url not in active_urls:
    tunnel = ngrok.connect(
        addr="http://127.0.0.1:8000",
        proto="http",
        bind_tls=True,
        inspect=False,
        pyngrok_config=ngrok_config,
    )

PUBLIC_BASE_URL = tunnel.public_url.rstrip("/") + "/v1"
print("Public URL:", tunnel.public_url)
print("EMBEDDING_BASE_URL:", PUBLIC_BASE_URL)

## 8. Thử endpoint ngrok rồi chờ khi bạn dùng từ local

Cell này gọi endpoint ngrok **một lần để kiểm tra**, rồi chờ tiến trình server. Không gửi request lặp để giữ phiên. Cell hiển thị đang chạy, nhưng Kaggle vẫn có thể dừng session theo giới hạn của nền tảng.

Khi muốn chạy cell khác, bấm **Interrupt**: chỉ ngắt phần chờ, không chủ động tắt vLLM/ngrok. Không chạy lại cell 5 khi server vẫn sống.


In [ ]:
response = http.post(
    PUBLIC_BASE_URL + "/embeddings", json=payload, headers=headers, timeout=(5, 60)
)
response.raise_for_status()
public_vector = response.json()["data"][0]["embedding"]
if len(public_vector) != 1024 or not all(math.isfinite(x) for x in public_vector):
    raise RuntimeError("Vector qua ngrok không hợp lệ.")
print("Endpoint ngrok OK. Dùng URL này ở local:", PUBLIC_BASE_URL, flush=True)
print("Đang chờ server. Interrupt để quay lại notebook; session vẫn chịu giới hạn Kaggle.", flush=True)
try:
    exit_code = server.wait()
    print("Server đã dừng, exit code:", exit_code)
except KeyboardInterrupt:
    print("Đã ngắt phần chờ; server và tunnel chưa bị chủ động dừng.")


### Khi dùng xong hoặc cần xem lỗi

Xem log trong cell mới:
```python
print(LOG.read_text(errors="replace")[-8000:])
```

Dừng dịch vụ do notebook này khởi động (chỉ chạy khi muốn tắt, sau khi Interrupt cell chờ):
```python
import signal
ngrok.disconnect(tunnel.public_url, pyngrok_config=ngrok_config)
tunnel = None
if server.poll() is None:
    os.killpg(server.pid, signal.SIGTERM)
    server.wait(timeout=30)
```

Nếu session bị dừng: mở phiên mới, chọn GPU/Internet, cấp quyền Secrets và chạy lại từ đầu. Không cần chạy dry-run. Nếu repo hoặc môi trường còn, các cell sẽ dùng lại chúng. Lấy URL mới từ output; revision model vẫn khóa trong script.
